In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report,confusion_matrix
import pickle

In [2]:
df = pd.read_csv("AI_Human.csv")


In [3]:
df.tail()

,text,generated
487230,Tie Face on Mars is really just a big misunder...,0.0
487231,The whole purpose of democracy is to create a ...,0.0
487232,I firmly believe that governments worldwide sh...,1.0
487233,I DFN't agree with this decision because a LFT...,0.0
487234,"Richard Non, Jimmy Carter, and Bob Dole and ot...",0.0


In [4]:
df.isnull().sum()

text         0
generated    0
dtype: int64

In [5]:
df = df.drop_duplicates(subset="text")

In [6]:
y = df["generated"]
x = df["text"]

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    x,y, test_size=0.2, random_state=42
)

In [8]:
x_train.shape

(389788,)

In [9]:
y_train.shape

(389788,)

In [10]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,3),
    max_features=50000,
    min_df=5
)

In [11]:
tfidf_x_train = tfidf.fit_transform(x_train)
tfidf_x_test = tfidf.transform(x_test)

In [12]:
model = LinearSVC(class_weight="balanced")

In [13]:
model.fit(tfidf_x_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,None


In [14]:
y_pred = model.predict(tfidf_x_test)

In [15]:
y_pred

array([0., 0., 0., ..., 0., 0., 1.], shape=(97447,))

In [16]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n         0.0       1.00      1.00      1.00     61112\n         1.0       1.00      1.00      1.00     36335\n\n    accuracy                           1.00     97447\n   macro avg       1.00      1.00      1.00     97447\nweighted avg       1.00      1.00      1.00     97447\n'

In [17]:
print(confusion_matrix(y_test, y_pred))

[[61088    24]
 [   51 36284]]


In [18]:
with open("text.pkl", "wb")as f:
    pickle.dump({
        "model":model,
        "tfidf":tfidf
    }, f)
    
print("Vocabulary size:", len(tfidf.vocabulary_))
print("model save successfully")

Vocabulary size: 50000
model save successfully


In [19]:
print(y_train.value_counts())

generated
0.0    244685
1.0    145103
Name: count, dtype: int64


In [20]:
print(model.classes_)

[0. 1.]


In [21]:
import numpy as np
np.unique(y_pred, return_counts=True)

(array([0., 1.]), array([61139, 36308]))

In [25]:
human_samples = df[df["generated"] == 0]["text"].head(2)

for i, text in enumerate(human_samples, 1):
    print(f"\nHuman Text {i}:\n{text}\n")


Human Text 1:
Cars. Cars have been around since they became famous in the 1900s, when Henry Ford created and built the first ModelT. Cars have played a major role in our every day lives since then. But now, people are starting to question if limiting car usage would be a good thing. To me, limiting the use of cars might be a good thing to do.

In like matter of this, article, "In German Suburb, Life Goes On Without Cars," by Elizabeth Rosenthal states, how automobiles are the linchpin of suburbs, where middle class families from either Shanghai or Chicago tend to make their homes. Experts say how this is a huge impediment to current efforts to reduce greenhouse gas emissions from tailpipe. Passenger cars are responsible for 12 percent of greenhouse gas emissions in Europe...and up to 50 percent in some carintensive areas in the United States. Cars are the main reason for the greenhouse gas emissions because of a lot of people driving them around all the time getting where they need to

In [26]:
ai_samples = df[df["generated"] == 1]["text"].head(3)

for i, text in enumerate(ai_samples, 1):
    print(f"\nAI Text {i}:\n{text}\n")


AI Text 1:
This essay will analyze, discuss and prove one reason in favor of keeping the Electoral College in the United States for its presidential elections. One of the reasons to keep the electoral college is that it is better for smaller, more rural states to have more influence as opposed to larger metropolitan areas that have large populations. The electors from these states are granted two votes each. Those from larger, more populated areas are granted just one vote each. Smaller states tend to hold significant power because their two votes for president and vice president add up more than the votes of larger states that have many electors. This is because of the split of the electoral votes. Some argue that electors are not bound to vote for the candidate who won the most votes nationally. They do not have to vote for their own state's nominee unless their state has a winner take all system. However, there are states that have adopted laws that force their electors to vote for